# PHAD-Net: Concordance-Discordance Network 
## Audio + Text Notebook

**Concordance-Discordance Decomposition (CDD)** — decomposes fused audio-text representations into concordant (both modalities agree) and discordant (modalities conflict) subspaces, letting the model explicitly reason about cases where prosody contradicts semantic content (e.g., calm delivery of hateful text, or aggressive tone with benign words).

**Architecture:** Wav2Vec2-XLS-R-300M (audio) + XLM-RoBERTa (text) → Cross-Modal Temporal Attention (CMTA) → Shared-Private encoders → CDD → Prosodic Dynamics Network (PDN) for pitch intensity contours → 5-branch gated fusion with modality dropout. Trained with focal loss + orthogonality regularization on shared-private subspaces.

In [1]:
import os
import warnings
from pathlib import Path
from typing import Dict, List, Tuple
from dataclasses import dataclass, field
from collections import defaultdict
import json

import numpy as np
import pandas as pd
import librosa
import torchaudio

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.cuda.amp import GradScaler, autocast

from transformers import Wav2Vec2Model, XLMRobertaModel, XLMRobertaTokenizer

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    import parselmouth
    from parselmouth.praat import call
except ImportError:
    os.system('pip install praat-parselmouth -q')
    import parselmouth
    from parselmouth.praat import call

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
# if torch.cuda.is_available():
#     print(f"GPU: {torch.cuda.get_device_name(0)}")
#     print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

Device: cuda


## Configuration

In [ ]:
@dataclass
class Config:
    base_path: str = "/workspace/ECHO"
    test_path: str = "/workspace/ECHO_Test"
    output_path: str = "/workspace/outputs_phad"

    languages: List[str] = field(default_factory=lambda: [
        "bengali", "chinese", "hindi", "english", "french",
        "japanese", "malayalam", "tamil", "telugu", "spanish"])

    random_seed: int = 42

    sample_rate: int = 16000
    max_duration_sec: float = 10.0
    max_samples: int = 160000

    text_model: str = "xlm-roberta-large"
    text_hidden_size: int = 768
    max_text_length: int = 128
    freeze_text_layers: int = 6

    pretrained_model: str = "facebook/wav2vec2-xls-r-300m"
    encoder_hidden_size: int = 1024
    freeze_encoder_layers: int = 21
    last_k_layers: int = 4

    hidden_dim: int = 128
    shared_dim: int = 128
    dropout: float = 0.35
    language_embed_dim: int = 32

    pdn_contour_length: int = 100
    pdn_channels: List[int] = field(default_factory=lambda: [32, 64, 64])
    pdn_kernel_sizes: List[int] = field(default_factory=lambda: [7, 5, 3])
    n_mfcc: int = 13

    cmta_heads: int = 4
    cmta_layers: int = 2

    modality_drop_prob: float = 0.15

    ortho_weight: float = 0.05

    batch_size: int = 8
    epochs: int = 80
    lr_pretrained: float = 1.5e-6
    lr_custom: float = 1.5e-4
    weight_decay: float = 0.03
    warmup_epochs: int = 8
    early_stopping_patience: int = 12
    gradient_accumulation: int = 4
    max_grad_norm: float = 1.0

    focal_gamma: float = 2.0
    label_smoothing: float = 0.05
    use_mixup: bool = True
    mixup_alpha: float = 0.3
    samples_per_language_per_batch: int = 2

    def __post_init__(self):
        self.max_samples = int(self.max_duration_sec * self.sample_rate)
        for d in [self.output_path, f"{self.output_path}/models",
                  f"{self.output_path}/features", f"{self.output_path}/results/figures"]:
            os.makedirs(d, exist_ok=True)

config = Config()
print(f"PHAD-Net v2 | Audio: {config.pretrained_model} | Text: {config.text_model}")
print(f"MDrop: {config.modality_drop_prob} | Ortho: {config.ortho_weight} | Eff batch: {config.batch_size * config.gradient_accumulation}")

PHAD-Net v2 | Audio: facebook/wav2vec2-xls-r-300m | Text: xlm-roberta-large
MDrop: 0.15 | Ortho: 0.05 | Eff batch: 32


## Data Preparation

In [3]:
class DataPreparator:
    def __init__(self, cfg):
        self.config = cfg
        self.all_metadata = {}
        self.label_encoder = LabelEncoder()

    @staticmethod
    def normalize_columns(df):
        df.columns = df.columns.str.lower().str.strip()
        for old, new in {'hate_label': 'label', 'hate_speech': 'label', 'hate': 'label',
                         'file_name': 'filename', 'audio_file': 'filename', 'file': 'filename',
                         'text': 'transcription', 'transcript': 'transcription', 'sentence': 'transcription'}.items():
            if old in df.columns and new not in df.columns:
                df = df.rename(columns={old: new})
        if 'gender' not in df.columns: df['gender'] = 'unknown'
        if 'transcription' not in df.columns: df['transcription'] = ''
        df['transcription'] = df['transcription'].fillna('').astype(str)
        return df

    def load_all_metadata(self):
        print("Loading metadata...")
        for lang in tqdm(self.config.languages):
            self.all_metadata[lang] = {}
            lp = Path(self.config.base_path) / lang
            for split in ['train', 'validation']:
                sp = lp / split / 'metadata.csv'
                if not sp.exists(): continue
                df = self.normalize_columns(pd.read_csv(sp))
                df['language'] = lang; df['split'] = split
                df['audio_path'] = df['filename'].apply(lambda x: str(lp / split / x))
                df = df[df['audio_path'].apply(lambda p: Path(p).exists())]
                self.all_metadata[lang][split] = df
                has_text = (df['transcription'].str.len() > 0).sum()
                print(f"  {lang}/{split}: {len(df)} files, {has_text} with text")

    def create_splits(self):
        print("\nCreating splits...")
        train_dfs, val_dfs = [], []
        for lang, splits in self.all_metadata.items():
            for sn, df in splits.items():
                if df.empty: continue
                (train_dfs if sn == 'train' else val_dfs).append(df)
        train_df = pd.concat(train_dfs, ignore_index=True)
        val_df = pd.concat(val_dfs, ignore_index=True)
        all_langs = pd.concat([train_df['language'], val_df['language']])
        self.label_encoder.fit(all_langs)
        for df in [train_df, val_df]:
            df['language_id'] = self.label_encoder.transform(df['language'])
        print(f"  Train: {len(train_df)}, Val: {len(val_df)}")
        return train_df, val_df

## Feature Extraction

In [4]:
class FeatureExtractor:
    def __init__(self, cfg):
        self.config = cfg

    def extract_prosody(self, audio_path):
        features = {}
        try:
            sound = parselmouth.Sound(audio_path)
            pitch = call(sound, "To Pitch", 0.0, 75, 600)
            pv = pitch.selected_array['frequency']; pv = pv[pv > 0]
            if len(pv) > 2:
                features['pitch_mean'] = np.mean(pv); features['pitch_std'] = np.std(pv)
                features['pitch_range'] = np.ptp(pv); features['pitch_median'] = np.median(pv)
                t = np.arange(len(pv), dtype=np.float64)
                features['pitch_slope'] = np.polyfit(t, pv, 1)[0] if len(t) > 1 else 0.0
                features['pitch_perturbation'] = np.mean(np.abs(np.diff(pv))) / (np.mean(pv) + 1e-8)
            else:
                for k in ['pitch_mean','pitch_std','pitch_range','pitch_median','pitch_slope','pitch_perturbation']:
                    features[k] = 0.0
            intensity = call(sound, "To Intensity", 100, 0.0)
            iv = np.array([call(intensity, "Get value at time", t, "cubic") for t in np.linspace(0, sound.duration, 80)])
            iv = iv[~np.isnan(iv)]
            if len(iv) > 2:
                features['intensity_mean'] = np.mean(iv); features['intensity_std'] = np.std(iv)
                features['intensity_range'] = np.ptp(iv)
                features['intensity_dyn_range'] = np.percentile(iv, 95) - np.percentile(iv, 5)
            else:
                for k in ['intensity_mean','intensity_std','intensity_range','intensity_dyn_range']: features[k] = 0.0
            features['voiced_fraction'] = call(pitch, "Count voiced frames") / max(pitch.n_frames, 1)
            pa = pitch.selected_array['frequency']; vb = (pa > 0).astype(int)
            features['speech_rate_proxy'] = (np.sum(np.abs(np.diff(vb))) // 2 + 1) / max(sound.duration, 0.1)
        except Exception:
            for k in ['pitch_mean','pitch_std','pitch_range','pitch_median','pitch_slope','pitch_perturbation',
                      'intensity_mean','intensity_std','intensity_range','intensity_dyn_range','voiced_fraction','speech_rate_proxy']:
                features[k] = 0.0
        return features

    def extract_voice_quality(self, audio_path):
        try:
            sound = parselmouth.Sound(audio_path)
            pp = call(sound, "To PointProcess (periodic, cc)", 75, 600)
            f = {'jitter': call(pp, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3),
                 'shimmer': call([sound, pp], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6),
                 'hnr': call(call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0), "Get mean", 0, 0)}
        except Exception:
            f = {'jitter': 0.0, 'shimmer': 0.0, 'hnr': 0.0}
        return {k: (0.0 if np.isnan(v) else float(v)) for k, v in f.items()}

    def extract_spectral(self, y, sr):
        f = {}
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=self.config.n_mfcc)
        md = librosa.feature.delta(mfccs); mdd = librosa.feature.delta(mfccs, order=2)
        for i in range(self.config.n_mfcc):
            f[f'mfcc_{i}_mean'] = float(np.mean(mfccs[i])); f[f'mfcc_{i}_std'] = float(np.std(mfccs[i]))
            f[f'mfcc_d_{i}_mean'] = float(np.mean(md[i])); f[f'mfcc_dd_{i}_mean'] = float(np.mean(mdd[i]))
        f['spectral_centroid'] = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
        f['spectral_bandwidth'] = float(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)))
        f['spectral_rolloff'] = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))
        f['spectral_flatness'] = float(np.mean(librosa.feature.spectral_flatness(y=y)))
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_bands=6)
        for i in range(contrast.shape[0]): f[f'spectral_contrast_{i}'] = float(np.mean(contrast[i]))
        rms = librosa.feature.rms(y=y)[0]
        f['rms_mean'] = float(np.mean(rms)); f['rms_std'] = float(np.std(rms))
        f['zcr'] = float(np.mean(librosa.feature.zero_crossing_rate(y)))
        oe = librosa.onset.onset_strength(y=y, sr=sr)
        f['onset_strength_mean'] = float(np.mean(oe)); f['onset_strength_std'] = float(np.std(oe))
        tempo = librosa.beat.tempo(onset_envelope=oe, sr=sr)
        f['tempo'] = float(tempo[0]) if len(tempo) > 0 else 0.0
        return f

    def extract_contours(self, y, sr, audio_path):
        L = self.config.pdn_contour_length
        c = np.zeros((4, L), dtype=np.float32)
        try:
            sound = parselmouth.Sound(audio_path)
            pitch = call(sound, "To Pitch", 0.0, 75, 600)
            pv = pitch.selected_array['frequency']
            if len(pv) > 1:
                v = pv > 0
                pi = np.interp(np.arange(len(pv)), np.where(v)[0], pv[v]) if v.any() else pv
                c[0] = np.interp(np.linspace(0, len(pi)-1, L), np.arange(len(pi)), pi)
        except: pass
        try:
            hop = int(0.010 * sr)
            for i, feat_fn in enumerate([lambda: librosa.feature.rms(y=y, hop_length=hop)[0],
                                         lambda: librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0],
                                         lambda: librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]]):
                vals = feat_fn()
                if len(vals) > 1:
                    c[i+1] = np.interp(np.linspace(0, len(vals)-1, L), np.arange(len(vals)), vals)
        except: pass
        return c

    def extract_all(self, audio_path):
        features, contours = {}, np.zeros((4, self.config.pdn_contour_length), dtype=np.float32)
        try:
            y, sr = librosa.load(audio_path, sr=self.config.sample_rate)
            if len(y) > self.config.max_samples: y = y[:self.config.max_samples]
            peak = np.max(np.abs(y))
            if peak > 0: y = y / peak
            features.update(self.extract_prosody(audio_path))
            features.update(self.extract_voice_quality(audio_path))
            features.update(self.extract_spectral(y, sr))
            contours = self.extract_contours(y, sr, audio_path)
        except: pass
        return features, contours

    def extract_from_dataframe(self, df, name):
        print(f"\nExtracting features for {name} ({len(df)} files)...")
        all_f, all_c = [], []
        for _, row in tqdm(df.iterrows(), total=len(df)):
            feats, cont = self.extract_all(row['audio_path'])
            feats.update({k: row[k] for k in ['filename','label','language','language_id','audio_path','transcription']})
            all_f.append(feats); all_c.append(cont)
        fdf = pd.DataFrame(all_f)
        fdf[fdf.select_dtypes(include=[np.number]).columns] = fdf.select_dtypes(include=[np.number]).fillna(0)
        return fdf, np.stack(all_c, axis=0)

## Model Components

In [5]:
class ProsodicDynamicsNetwork(nn.Module):
    def __init__(self, in_channels=4, channels=[32,64,64], kernel_sizes=[7,5,3], output_dim=128, dropout=0.3):
        super().__init__()
        layers = []; prev = in_channels
        for ch, ks in zip(channels, kernel_sizes):
            layers.extend([nn.Conv1d(prev, ch, ks, padding=ks//2), nn.BatchNorm1d(ch), nn.GELU(), nn.Dropout(dropout)])
            prev = ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.Linear(channels[-1], 1)
        self.proj = nn.Sequential(nn.Linear(channels[-1], output_dim), nn.LayerNorm(output_dim), nn.GELU())
    def forward(self, x):
        x = self.conv(x).transpose(1, 2)
        w = F.softmax(self.pool(x).squeeze(-1), dim=1).unsqueeze(-1)
        return self.proj((x * w).sum(dim=1))

class CrossModalTemporalAttention(nn.Module):
    def __init__(self, audio_dim, text_dim, hidden_dim, num_heads=4, num_layers=2, dropout=0.3):
        super().__init__()
        self.audio_proj = nn.Linear(audio_dim, hidden_dim)
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.a2t_layers = nn.ModuleList([nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True) for _ in range(num_layers)])
        self.a2t_norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])
        self.t2a_layers = nn.ModuleList([nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True) for _ in range(num_layers)])
        self.t2a_norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
    def forward(self, audio_seq, text_seq, text_mask=None):
        a = self.audio_proj(audio_seq); t = self.text_proj(text_seq)
        kpm = (text_mask == 0) if text_mask is not None else None
        for attn, norm in zip(self.a2t_layers, self.a2t_norms):
            res = a; a, _ = attn(a, t, t, key_padding_mask=kpm); a = norm(res + self.dropout(a))
        for attn, norm in zip(self.t2a_layers, self.t2a_norms):
            res = t; t, _ = attn(t, a, a); t = norm(res + self.dropout(t))
        audio_ctx = a.mean(dim=1)
        if text_mask is not None:
            me = text_mask.unsqueeze(-1).float(); text_ctx = (t * me).sum(1) / (me.sum(1) + 1e-8)
        else: text_ctx = t.mean(dim=1)
        return audio_ctx, text_ctx

class SharedPrivateEncoder(nn.Module):
    def __init__(self, input_dim, shared_dim, private_dim, dropout=0.3):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(input_dim, shared_dim), nn.LayerNorm(shared_dim), nn.GELU(), nn.Dropout(dropout))
        self.private = nn.Sequential(nn.Linear(input_dim, private_dim), nn.LayerNorm(private_dim), nn.GELU(), nn.Dropout(dropout))
    def forward(self, x): return self.shared(x), self.private(x)

def orthogonality_loss(shared, private):
    sn = F.normalize(shared, dim=1); pn = F.normalize(private, dim=1)
    return torch.norm(sn.t() @ pn, p='fro') ** 2 / shared.size(0)

class ConcordanceDiscordanceDecomposition(nn.Module):
    def __init__(self, shared_dim, output_dim, dropout=0.3):
        super().__init__()
        self.conc_net = nn.Sequential(nn.Linear(shared_dim, output_dim), nn.LayerNorm(output_dim), nn.GELU(), nn.Dropout(dropout))
        self.disc_net = nn.Sequential(nn.Linear(shared_dim, output_dim), nn.LayerNorm(output_dim), nn.GELU(), nn.Dropout(dropout))
        self.balance = nn.Sequential(nn.Linear(output_dim * 2, 2), nn.Softmax(dim=-1))
    def forward(self, shared_a, shared_t):
        conc = self.conc_net(shared_a * shared_t)
        disc = self.disc_net(torch.abs(shared_a - shared_t))
        gate = self.balance(torch.cat([conc, disc], dim=-1))
        fused = gate[:, 0:1] * conc + gate[:, 1:2] * disc
        return fused, conc, disc

class GatedFusion(nn.Module):
    def __init__(self, dim, num_branches, dropout=0.3):
        super().__init__()
        total = dim * num_branches
        self.gate = nn.Sequential(nn.Linear(total, dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim, num_branches), nn.Sigmoid())
        self.proj = nn.Sequential(nn.Linear(total, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(dropout))
    def forward(self, branches):
        cat = torch.cat(branches, dim=1); g = self.gate(cat)
        gated = torch.cat([b * g[:, i:i+1] for i, b in enumerate(branches)], dim=1)
        return self.proj(gated)

## PHAD-Net v2 Model

In [6]:
class PHADNetV2(nn.Module):
    def __init__(self, num_handcrafted_features, num_languages, cfg):
        super().__init__()
        self.config = cfg; dim = cfg.hidden_dim; sdim = cfg.shared_dim

        # Audio encoder
        print(f"Loading {cfg.pretrained_model}...")
        self.audio_encoder = Wav2Vec2Model.from_pretrained(cfg.pretrained_model, output_hidden_states=True)
        audio_dim = self.audio_encoder.config.hidden_size
        for p in self.audio_encoder.feature_extractor.parameters(): p.requires_grad = False
        for p in self.audio_encoder.feature_projection.parameters(): p.requires_grad = False
        for i, layer in enumerate(self.audio_encoder.encoder.layers):
            if i < cfg.freeze_encoder_layers:
                for p in layer.parameters(): p.requires_grad = False

        self.audio_pool = nn.Sequential(nn.Linear(audio_dim, dim), nn.Tanh(), nn.Linear(dim, 1))
        self.audio_proj = nn.Sequential(nn.Linear(audio_dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(cfg.dropout))

        # Text encoder
        print(f"Loading {cfg.text_model}...")
        self.text_encoder = XLMRobertaModel.from_pretrained(cfg.text_model)
        text_dim = self.text_encoder.config.hidden_size
        for p in self.text_encoder.embeddings.parameters(): p.requires_grad = False
        for i, layer in enumerate(self.text_encoder.encoder.layer):
            if i < cfg.freeze_text_layers:
                for p in layer.parameters(): p.requires_grad = False

        self.text_proj = nn.Sequential(nn.Linear(text_dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(cfg.dropout))

        self.cmta = CrossModalTemporalAttention(audio_dim, text_dim, dim, cfg.cmta_heads, cfg.cmta_layers, cfg.dropout)
        self.audio_sp = SharedPrivateEncoder(dim, sdim, dim, cfg.dropout)
        self.text_sp = SharedPrivateEncoder(dim, sdim, dim, cfg.dropout)
        self.cdd = ConcordanceDiscordanceDecomposition(sdim, dim, cfg.dropout)

        self.pdn = ProsodicDynamicsNetwork(4, cfg.pdn_channels, cfg.pdn_kernel_sizes, dim//2, cfg.dropout)
        self.hcf = nn.Sequential(nn.Linear(num_handcrafted_features, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(cfg.dropout),
                                 nn.Linear(dim, dim//2), nn.LayerNorm(dim//2), nn.GELU(), nn.Dropout(cfg.dropout))
        self.acoustic_combine = nn.Sequential(nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(cfg.dropout))

        self.lang_embed = nn.Embedding(num_languages, cfg.language_embed_dim)
        self.fusion = GatedFusion(dim, 5, cfg.dropout)
        self.classifier = nn.Sequential(
            nn.Linear(dim + cfg.language_embed_dim, dim), nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(dim, dim//2), nn.GELU(), nn.Dropout(cfg.dropout * 0.5), nn.Linear(dim//2, 2))

    def forward(self, waveform, input_ids, attention_mask, handcrafted, contours, language_ids):
        # Audio
        aout = self.audio_encoder(waveform, output_hidden_states=True)
        audio_seq = torch.stack(aout.hidden_states[-self.config.last_k_layers:], dim=0).mean(0)
        a_attn = F.softmax(self.audio_pool(audio_seq).squeeze(-1), dim=1)
        audio_repr = self.audio_proj((audio_seq * a_attn.unsqueeze(-1)).sum(1))

        # Text
        tout = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_seq = tout.last_hidden_state
        text_repr = self.text_proj(text_seq[:, 0])

        # CMTA
        audio_ctx, text_ctx = self.cmta(audio_seq, text_seq, attention_mask)
        audio_enriched = audio_repr + audio_ctx
        text_enriched = text_repr + text_ctx

        # Modality Dropout
        if self.training:
            p = self.config.modality_drop_prob
            drop_a = np.random.random() < p; drop_t = np.random.random() < p
            if drop_a and drop_t:
                if np.random.random() < 0.5: drop_a = False
                else: drop_t = False
            if drop_a: audio_enriched = torch.zeros_like(audio_enriched)
            if drop_t: text_enriched = torch.zeros_like(text_enriched)
            n_active = (not drop_a) + (not drop_t)
            scale = 2.0 / n_active
            audio_enriched = audio_enriched * scale; text_enriched = text_enriched * scale

        # Shared-Private
        audio_shared, audio_private = self.audio_sp(audio_enriched)
        text_shared, text_private = self.text_sp(text_enriched)

        # CDD
        cdd_repr, concordance, discordance = self.cdd(audio_shared, text_shared)

        # PDN + handcrafted
        acoustic = self.acoustic_combine(torch.cat([self.pdn(contours), self.hcf(handcrafted)], dim=1))

        # 5-branch fusion
        fused = self.fusion([audio_private, text_private, concordance, discordance, acoustic])
        combined = torch.cat([fused, self.lang_embed(language_ids)], dim=1)
        logits = self.classifier(combined)

        return {'logits': logits, 'fused': fused,
                'audio_shared': audio_shared, 'audio_private': audio_private,
                'text_shared': text_shared, 'text_private': text_private,
                'concordance': concordance, 'discordance': discordance}

## Dataset & DataLoader

In [7]:
class MultimodalDataset(Dataset):
    def __init__(self, fdf, cont, feat_cols, tokenizer, scaler=None, cscaler=None,
                 max_samples=160000, max_text_len=128, augment=False):
        self.df = fdf.reset_index(drop=True)
        self.max_samples = max_samples; self.max_text_len = max_text_len
        self.augment = augment; self.tokenizer = tokenizer
        self.features = self.df[feat_cols].values.astype(np.float32)
        if scaler: self.features = scaler.transform(self.features)
        self.contours = cont.astype(np.float32)
        if cscaler:
            for c in range(4):
                m, s = cscaler[c]
                if s > 0: self.contours[:, c, :] = (self.contours[:, c, :] - m) / s
        self.labels = self.df['label'].values.astype(np.int64)
        self.lang_ids = self.df['language_id'].values.astype(np.int64)
        self.audio_paths = self.df['audio_path'].values
        self.texts = self.df['transcription'].values
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        wav, sr = torchaudio.load(self.audio_paths[idx])
        if sr != 16000: wav = torchaudio.transforms.Resample(sr, 16000)(wav)
        if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
        wav = wav.squeeze(0)
        if len(wav) > self.max_samples:
            s = np.random.randint(0, len(wav)-self.max_samples) if self.augment else 0
            wav = wav[s:s+self.max_samples]
        elif len(wav) < self.max_samples: wav = F.pad(wav, (0, self.max_samples - len(wav)))
        if self.augment:
            if np.random.random() < 0.3: wav = wav + np.random.uniform(0.001, 0.008) * torch.randn_like(wav)
            if np.random.random() < 0.3: wav = wav * np.random.uniform(0.8, 1.2)
        enc = self.tokenizer(str(self.texts[idx]), max_length=self.max_text_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return (wav, enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0),
                torch.tensor(self.features[idx], dtype=torch.float32),
                torch.tensor(self.contours[idx], dtype=torch.float32),
                self.labels[idx], self.lang_ids[idx])

class LanguageBalancedSampler(Sampler):
    def __init__(self, lang_ids, labels, spl=2, nb=None):
        ul = np.unique(lang_ids); self.nl = len(ul)
        self.lci = {l: {c: np.where((lang_ids==l)&(labels==c))[0] for c in [0,1]} for l in ul}
        self.spl = spl; self.bs = spl * self.nl
        self.nb = nb or max(1, len(lang_ids)//self.bs)
    def __iter__(self):
        idx = []
        for _ in range(self.nb):
            for l in self.lci:
                for i in range(self.spl):
                    c = i%2; pool = self.lci[l][c]
                    if len(pool)==0: pool = self.lci[l][1-c]
                    if len(pool)>0: idx.append(np.random.choice(pool))
        return iter(idx)
    def __len__(self): return self.nb * self.bs

def compute_contour_stats(ca):
    return [(float(np.mean(ca[:, c])), float(np.std(ca[:, c]) + 1e-8)) for c in range(ca.shape[1])]

def create_data_loaders(tf, tc, vf, vc, tokenizer, cfg):
    meta = ['filename','label','language','language_id','audio_path','gender','transcription']
    fc = [c for c in tf.columns if c not in meta and c not in ['label','language_id']
          and tf[c].dtype in [np.float64, np.float32, np.int64]]
    print(f"Handcrafted features: {len(fc)}")
    scaler = StandardScaler(); scaler.fit(tf[fc].values)
    cs = compute_contour_stats(tc)
    kw = dict(feat_cols=fc, tokenizer=tokenizer, scaler=scaler, cscaler=cs,
              max_samples=cfg.max_samples, max_text_len=cfg.max_text_length)
    trd = MultimodalDataset(tf, tc, augment=True, **kw)
    vd = MultimodalDataset(vf, vc, augment=False, **kw)
    sam = LanguageBalancedSampler(tf['language_id'].values, tf['label'].values, cfg.samples_per_language_per_batch)
    return (DataLoader(trd, batch_size=cfg.batch_size, sampler=sam, num_workers=2, pin_memory=True, drop_last=True),
            DataLoader(vd, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True),
            scaler, cs, fc)

## Loss & Training

In [8]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.gamma = gamma; self.smoothing = smoothing
    def forward(self, inputs, targets):
        nc = inputs.size(1)
        with torch.no_grad():
            sm = torch.zeros_like(inputs).scatter_(1, targets.unsqueeze(1), 1.0)
            sm = sm * (1-self.smoothing) + self.smoothing/nc
        lp = F.log_softmax(inputs, dim=1); p = torch.exp(lp)
        return (-(sm * (1-p)**self.gamma * lp).sum(1)).mean()

class Trainer:
    def __init__(self, model, cfg, device):
        self.model = model; self.config = cfg; self.device = device
        pre, cust = [], []
        for n, p in model.named_parameters():
            if not p.requires_grad: continue
            (pre if n.startswith('audio_encoder.') or n.startswith('text_encoder.') else cust).append(p)
        self.optimizer = optim.AdamW([{'params': pre, 'lr': cfg.lr_pretrained},
                                      {'params': cust, 'lr': cfg.lr_custom}], weight_decay=cfg.weight_decay)
        self.criterion = FocalLoss(cfg.focal_gamma, cfg.label_smoothing)
        self.scaler = GradScaler()
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(self.optimizer, T_0=15, T_mult=2, eta_min=1e-6)
        self.history = defaultdict(list); self.best_f1 = 0; self.patience = 0

    def warmup_lr(self, epoch):
        if epoch < self.config.warmup_epochs:
            f = (epoch+1)/self.config.warmup_epochs
            self.optimizer.param_groups[0]['lr'] = self.config.lr_pretrained * f
            self.optimizer.param_groups[1]['lr'] = self.config.lr_custom * f

    def train_epoch(self, loader, epoch):
        self.model.train(); tot, tot_cls, tot_ort = 0, 0, 0; ap, al = [], []
        self.optimizer.zero_grad()
        for i, (wav, ids, mask, feat, cont, lab, lang) in enumerate(tqdm(loader, desc=f"Train E{epoch+1}")):
            wav, ids, mask = wav.to(self.device), ids.to(self.device), mask.to(self.device)
            feat, cont = feat.to(self.device), cont.to(self.device)
            lab, lang = lab.to(self.device), lang.to(self.device)
            with autocast():
                out = self.model(wav, ids, mask, feat, cont, lang)
                cls_loss = self.criterion(out['logits'], lab)
                ortho = (orthogonality_loss(out['audio_shared'], out['audio_private']) +
                         orthogonality_loss(out['text_shared'], out['text_private']))
                loss = (cls_loss + self.config.ortho_weight * ortho) / self.config.gradient_accumulation
            self.scaler.scale(loss).backward()
            if (i+1) % self.config.gradient_accumulation == 0:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)
                self.scaler.step(self.optimizer); self.scaler.update(); self.optimizer.zero_grad()
            tot += loss.item()*self.config.gradient_accumulation; tot_cls += cls_loss.item(); tot_ort += ortho.item()
            al.extend(lab.cpu().numpy()); ap.extend(out['logits'].argmax(1).cpu().numpy())
        n = max(len(loader),1)
        return {'loss': tot/n, 'cls_loss': tot_cls/n, 'ortho_loss': tot_ort/n,
                'f1': f1_score(al, ap, zero_division=0), 'acc': accuracy_score(al, ap)}

    @torch.no_grad()
    def validate(self, loader):
        self.model.eval(); tot = 0; ap, al, aprob = [], [], []
        for wav, ids, mask, feat, cont, lab, lang in loader:
            wav, ids, mask = wav.to(self.device), ids.to(self.device), mask.to(self.device)
            feat, cont = feat.to(self.device), cont.to(self.device)
            lab, lang = lab.to(self.device), lang.to(self.device)
            out = self.model(wav, ids, mask, feat, cont, lang)
            tot += self.criterion(out['logits'], lab).item()
            aprob.extend(F.softmax(out['logits'],1)[:,1].cpu().numpy())
            ap.extend(out['logits'].argmax(1).cpu().numpy()); al.extend(lab.cpu().numpy())
        try: auc = roc_auc_score(al, aprob)
        except: auc = 0.5
        return {'loss': tot/max(len(loader),1), 'f1': f1_score(al, ap, zero_division=0),
                'precision': precision_score(al, ap, zero_division=0),
                'recall': recall_score(al, ap, zero_division=0), 'acc': accuracy_score(al, ap), 'auc': auc}

    def train(self, tl, vl):
        print(f"\n{'='*60}\nTRAINING PHAD-Net v2 (Audio+Text)\n{'='*60}")
        for epoch in range(self.config.epochs):
            self.warmup_lr(epoch); tm = self.train_epoch(tl, epoch); vm = self.validate(vl)
            if epoch >= self.config.warmup_epochs: self.scheduler.step(epoch - self.config.warmup_epochs)
            gap = tm['f1'] - vm['f1']; lr = self.optimizer.param_groups[1]['lr']
            print(f"\nEpoch {epoch+1}/{self.config.epochs} | LR: {lr:.2e}")
            print(f"  Train — cls: {tm['cls_loss']:.4f}, ortho: {tm['ortho_loss']:.4f}, F1: {tm['f1']:.4f}, Acc: {tm['acc']:.4f}")
            print(f"  Val   — loss: {vm['loss']:.4f}, F1: {vm['f1']:.4f}, P: {vm['precision']:.3f}, R: {vm['recall']:.3f}, AUC: {vm['auc']:.4f}, Acc: {vm['acc']:.4f}")
            print(f"  Gap: {gap:.4f}")
            for k, v in tm.items(): self.history[f'train_{k}'].append(v)
            for k, v in vm.items(): self.history[f'val_{k}'].append(v)
            if vm['f1'] > self.best_f1:
                self.best_f1 = vm['f1']; self.patience = 0
                torch.save({'epoch': epoch, 'model_state_dict': self.model.state_dict(), 'val_f1': self.best_f1},
                           f"{self.config.output_path}/models/best_model.pt")
                print(f"  ✓ New best (F1: {self.best_f1:.4f})")
            else: self.patience += 1
            if self.patience >= self.config.early_stopping_patience:
                print(f"\nEarly stopping at epoch {epoch+1}"); break
            torch.cuda.empty_cache()
        ckpt = f"{self.config.output_path}/models/best_model.pt"
        if os.path.exists(ckpt):
            s = torch.load(ckpt, map_location=self.device)
            self.model.load_state_dict(s['model_state_dict'])
            print(f"Loaded best from epoch {s['epoch']+1}")
        return self.model, dict(self.history)

## Validation Thresholds & Inference

In [ ]:
def compute_val_thresholds(model, val_loader, val_df, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for wav, ids, mask, feat, cont, lab, lang in val_loader:
            wav, ids, mask = wav.to(device), ids.to(device), mask.to(device)
            feat, cont, lang = feat.to(device), cont.to(device), lang.to(device)
            out = model(wav, ids, mask, feat, cont, lang)
            all_probs.extend(F.softmax(out['logits'], 1)[:, 1].cpu().numpy())
            all_labels.extend(lab.numpy())
    all_probs, all_labels = np.array(all_probs), np.array(all_labels)
    languages = val_df['language'].values[:len(all_labels)]
    thresholds = {}
    for lang in np.unique(languages):
        m = languages == lang
        if m.sum() > 5:
            best_t, best_f1 = 0.5, 0
            for t in np.arange(0.25, 0.75, 0.02):
                f = f1_score(all_labels[m], (all_probs[m] >= t).astype(int), zero_division=0)
                if f > best_f1: best_f1, best_t = f, t
            thresholds[lang] = best_t
            print(f"  {lang:12s}: τ={best_t:.2f} (F1={best_f1:.3f})")
        else: thresholds[lang] = 0.5
    return thresholds


def run_inference(model, tokenizer, config, label_encoder, scaler, contour_stats, feat_cols, thresholds):
    model.eval()
    test_path = Path(config.test_path)
    extractor = FeatureExtractor(config)
    print(f"\n{'='*60}\nINFERENCE on {test_path}\n{'='*60}")

    for lang in config.languages:
        meta_path = test_path / lang / 'metadata.csv'
        if not meta_path.exists():
            print(f"  {lang}: skipping"); continue

        df = pd.read_csv(meta_path)
        df.columns = df.columns.str.lower().str.strip()

        audio_col = None
        for c in ['audio', 'filename', 'file_name', 'file', 'audio_file', 'path']:
            if c in df.columns: audio_col = c; break
        if audio_col is None:
            print(f"  {lang}: no audio column found in {list(df.columns)}, skipping"); continue
        #audio_col = 'audio'
        # for c in ['filename', 'file_name', 'file']:
        #     if c in df.columns: audio_col = c; break
        for old, new in {'text': 'transcription', 'transcript': 'transcription', 'sentence': 'transcription'}.items():
            if old in df.columns and new not in df.columns: df = df.rename(columns={old: new})
        text_col = 'transcription' if 'transcription' in df.columns else 'text'
        df[text_col] = df[text_col].fillna('').astype(str)
        df['audio_path'] = df[audio_col].apply(lambda x: str(test_path / lang / x))

        lang_id = label_encoder.transform([lang])[0]
        thresh = thresholds.get(lang, 0.5)

        all_features, all_contours = [], []
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"  {lang} features"):
            feats, cont = extractor.extract_all(row['audio_path'])
            all_features.append(feats); all_contours.append(cont)

        feat_df = pd.DataFrame(all_features)
        for c in feat_cols:
            if c not in feat_df.columns: feat_df[c] = 0.0
        feat_vals = scaler.transform(feat_df[feat_cols].fillna(0).values).astype(np.float32)

        contours_arr = np.stack(all_contours, axis=0).astype(np.float32)
        for c in range(4):
            m, s = contour_stats[c]
            if s > 0: contours_arr[:, c, :] = (contours_arr[:, c, :] - m) / s

        all_probs = []
        bs = config.batch_size
        for start in range(0, len(df), bs):
            end = min(start + bs, len(df))

            wavs = []
            for idx in range(start, end):
                wav, sr = torchaudio.load(df['audio_path'].values[idx])
                if sr != 16000: wav = torchaudio.transforms.Resample(sr, 16000)(wav)
                if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
                wav = wav.squeeze(0)
                if len(wav) > config.max_samples: wav = wav[:config.max_samples]
                elif len(wav) < config.max_samples: wav = F.pad(wav, (0, config.max_samples - len(wav)))
                wavs.append(wav)

            texts = list(df[text_col].values[start:end])
            enc = tokenizer(texts, max_length=config.max_text_length,
                           padding='max_length', truncation=True, return_tensors='pt')

            wav_b = torch.stack(wavs).to(DEVICE)
            ids_b = enc['input_ids'].to(DEVICE)
            mask_b = enc['attention_mask'].to(DEVICE)
            feat_b = torch.tensor(feat_vals[start:end]).to(DEVICE)
            cont_b = torch.tensor(contours_arr[start:end]).to(DEVICE)
            lang_b = torch.full((end-start,), lang_id, dtype=torch.long, device=DEVICE)

            with torch.no_grad():
                out = model(wav_b, ids_b, mask_b, feat_b, cont_b, lang_b)
                probs = F.softmax(out['logits'], dim=1)[:, 1].cpu().numpy()
                all_probs.extend(probs)

        df['label'] = (np.array(all_probs) >= thresh).astype(int)
        out_path = f"{config.output_path}/{lang}.csv"
        df.to_csv(out_path, index=False)
        print(f"  {lang:12s}: {len(df)} samples, τ={thresh:.2f}, {df['label'].mean()*100:.1f}% hate → {out_path}")

## Pipeline

In [10]:
def run_pipeline(config):
    print("=" * 70)
    print("PHAD-Net v2 (Audio + Text)")
    print("=" * 70)

    prep = DataPreparator(config)
    prep.load_all_metadata()
    train_df, val_df = prep.create_splits()

    ext = FeatureExtractor(config)
    tf, tc = ext.extract_from_dataframe(train_df, "train")
    vf, vc = ext.extract_from_dataframe(val_df, "val")
    for name, (f, c) in [('train',(tf,tc)),('val',(vf,vc))]:
        f.to_csv(f"{config.output_path}/features/{name}.csv", index=False)
        np.save(f"{config.output_path}/features/{name}_contours.npy", c)

    tokenizer = XLMRobertaTokenizer.from_pretrained(config.text_model)
    train_loader, val_loader, scaler, cstats, feat_cols = create_data_loaders(
        tf, tc, vf, vc, tokenizer, config)

    model = PHADNetV2(len(feat_cols), len(config.languages), config).to(DEVICE)
    nt = sum(p.numel() for p in model.parameters())
    nr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Params: {nt:,} total, {nr:,} trainable ({nr/nt*100:.1f}%)")

    trainer = Trainer(model, config, DEVICE)
    model, history = trainer.train(train_loader, val_loader)
    with open(f"{config.output_path}/models/history.json", 'w') as f: json.dump(history, f, indent=2)

    print("\nComputing per-language thresholds on validation...")
    thresholds = compute_val_thresholds(model, val_loader, vf, DEVICE)

    run_inference(model, tokenizer, config, prep.label_encoder, scaler, cstats, feat_cols, thresholds)

    return model, history

In [11]:
if not os.path.exists(config.base_path):
    print(f"Dataset not found at {config.base_path}")
else:
    model, history = run_pipeline(config)

PHAD-Net v2 (Audio + Text)
Loading metadata...


  0%|          | 0/10 [00:00<?, ?it/s]

  bengali/train: 1096 files, 1096 with text
  bengali/validation: 200 files, 200 with text
  chinese/train: 626 files, 626 with text
  chinese/validation: 134 files, 134 with text
  hindi/train: 1100 files, 1100 with text
  hindi/validation: 200 files, 200 with text
  english/train: 1096 files, 1096 with text
  english/validation: 199 files, 199 with text
  french/train: 767 files, 767 with text
  french/validation: 164 files, 164 with text
  japanese/train: 350 files, 350 with text
  japanese/validation: 75 files, 75 with text
  malayalam/train: 618 files, 618 with text
  malayalam/validation: 132 files, 132 with text
  tamil/train: 356 files, 356 with text
  tamil/validation: 76 files, 76 with text
  telugu/train: 385 files, 385 with text
  telugu/validation: 82 files, 82 with text
  spanish/train: 870 files, 870 with text
  spanish/validation: 186 files, 186 with text

Creating splits...
  Train: 7264, Val: 1448

Extracting features for train (7264 files)...


  0%|          | 0/7264 [00:00<?, ?it/s]


Extracting features for val (1448 files)...


...

  tamil       : 77 samples, τ=0.25, 50.6% hate → /workspace/outputs_phad/tamil.csv


  telugu features:   0%|          | 0/84 [00:00<?, ?it/s]

  telugu      : 84 samples, τ=0.71, 63.1% hate → /workspace/outputs_phad/telugu.csv


  spanish features:   0%|          | 0/187 [00:00<?, ?it/s]

  spanish     : 187 samples, τ=0.73, 34.8% hate → /workspace/outputs_phad/spanish.csv
